# Milestone 1 — Dataset Exploration (EDA)

This notebook performs exploratory data analysis (EDA) on the Biohub Cell Tracking dataset. We use the custom `ZarrDatasetReader` class to load the Zarr v3 datasets, query shape, data types, voxel sizes, compute statistics, and visualize orthogonal slices.

## 1. Load Libraries

In [ ]:
%matplotlib inline
import sys
import json
from pathlib import Path

# Add project root to sys.path to allow importing src
project_root = str(Path("..").resolve())
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import matplotlib.pyplot as plt
import numpy as np
import zarr
from src.datasets.zarr_dataset import ZarrDatasetReader

# Set style for visualization
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Libraries loaded successfully.")

## 2. Load First Dataset Automatically

In [ ]:
train_dir = Path("../data/raw/train")
zarr_paths = sorted(list(train_dir.glob("*.zarr")))

if not zarr_paths:
    raise FileNotFoundError("No .zarr files found in data/raw/train/")

first_dataset_path = zarr_paths[0]
print(f"Selected first dataset: {first_dataset_path.name}")

reader = ZarrDatasetReader(first_dataset_path)

## 3. Display Dataset Metadata

In [ ]:
shape = reader.get_shape()
dtype = reader.get_dtype()
voxel_size = reader.get_voxel_size()

T, Z, Y, X = shape

print(f"Dataset Name:  {first_dataset_path.name}")
print(f"Shape (T,Z,Y,X): {shape}")
print(f"Data Type:      {dtype}")
print(f"Voxel Size:     {voxel_size} (Z, Y, X) micrometers")
print(f"Timepoints:     {T}")
print(f"Image Size:     {Z}x{Y}x{X} (Z, Y, X)")

## 4. Show Orthogonal Slices (XY, XZ, YZ)

In [ ]:
# Use the middle indices for visualization
mid_t = T // 2
mid_z = Z // 2
mid_y = Y // 2
mid_x = X // 2

slice_xy = reader.get_slice_xy(mid_t, mid_z)
slice_xz = reader.get_slice_xz(mid_t, mid_y)
slice_yz = reader.get_slice_yz(mid_t, mid_x)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(slice_xy, cmap='gray')
axes[0].set_title(f"XY Slice (t={mid_t}, z={mid_z})")
axes[0].axis('off')

axes[1].imshow(slice_xz, cmap='gray')
axes[1].set_title(f"XZ Slice (t={mid_t}, y={mid_y})")
axes[1].axis('off')

axes[2].imshow(slice_yz, cmap='gray')
axes[2].set_title(f"YZ Slice (t={mid_t}, x={mid_x})")
axes[2].axis('off')

plt.tight_layout()
plt.show()

## 5. Show Histogram of Pixel Intensities

In [ ]:
# Compute histogram of a single frame (middle timepoint, middle Z)
frame_sample = reader.get_slice_xy(mid_t, mid_z)

plt.figure(figsize=(10, 5))
plt.hist(frame_sample.ravel(), bins=100, color='blue', alpha=0.7, log=True)
plt.title(f"Pixel Intensity Distribution (Log Scale, t={mid_t}, z={mid_z})")
plt.xlabel("Pixel Value")
plt.ylabel("Count (Log Scale)")
plt.grid(True, which="both", ls="--")
plt.show()

## 6. Dataset Statistics

In [ ]:
print("Computing statistics over the entire dataset (this may take a few seconds)...")
stats = reader.get_statistics()

print(f"Min Value:  {stats['min']}")
print(f"Max Value:  {stats['max']}")
print(f"Mean Value: {stats['mean']:.4f}")
print(f"Std Dev:    {stats['std']:.4f}")

## 7. Metadata raw content from zarr.json

In [ ]:
metadata_file = first_dataset_path / "zarr.json"
with open(metadata_file, "r", encoding="utf-8") as f:
    metadata_dict = json.load(f)

print(json.dumps(metadata_dict, indent=2))

## 8. Interactive Slice Viewer

In [ ]:
try:
    import ipywidgets as widgets
    from ipywidgets import interact
    HAS_IPYWIDGETS = True
except ImportError:
    HAS_IPYWIDGETS = False

if HAS_IPYWIDGETS:
    @interact(t=(0, T - 1), z=(0, Z - 1))
    def interactive_plot(t=mid_t, z=mid_z):
        slice_xy = reader.get_slice_xy(t, z)
        plt.figure(figsize=(7, 7))
        plt.imshow(slice_xy, cmap='gray')
        plt.title(f"Interactive XY Slice (t={t}, z={z})")
        plt.axis('off')
        plt.show()
else:
    print("ipywidgets not installed. Showing static middle slice instead.")
    slice_xy = reader.get_slice_xy(mid_t, mid_z)
    plt.figure(figsize=(7, 7))
    plt.imshow(slice_xy, cmap='gray')
    plt.title(f"Static XY Slice (t={mid_t}, z={mid_z})")
    plt.axis('off')
    plt.show()